<a href="https://colab.research.google.com/github/zafar-equbal/Data-structures-and-algorithms/blob/main/notebooks/FactCheck_AI_DistilBERT_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.14.1+cu130
GPU available: True
GPU name: Tesla T4


In [2]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy

In [5]:
!git clone https://github.com/zafar-equbal/FactCheck-AI

Cloning into 'FactCheck-AI'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 42 (delta 7), reused 40 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 626.79 KiB | 2.01 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [6]:
import os

os.chdir("/content/FactCheck-AI")

print("Current directory:", os.getcwd())
print("\nProject files:")

for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if d not in ["venv", "node_modules", ".git", "__pycache__"]]
    level = root.count(os.sep)
    print("  " * level + os.path.basename(root) + "/")

    for file in files:
        print("  " * (level + 1) + file)

Current directory: /content/FactCheck-AI

Project files:
./
  README.md
  .gitignore
  frontend/
    package-lock.json
    vite.config.js
    package.json
    index.html
    src/
      App.jsx
      styles.css
      main.jsx
  backend/
    requirements.txt
    analyze_dataset.py
    test_model.py
    prepare_dataset.py
    data/
      news.csv
    models/
      news_pipeline.joblib
      .gitkeep
    app/
      main.py
      __init__.py
      experiment_models.py
      train_news_model.py


In [7]:
with open("backend/prepare_dataset.py", "r", encoding="utf-8") as file:
    print(file.read())


from pathlib import Path

import pandas as pd

# Locate the backend directory
BASE_DIR = Path(__file__).resolve().parent
DATA_DIR = BASE_DIR / "data"

INPUT_FILE = DATA_DIR / "train-00000-of-00001-290868f0a36350c5.parquet"
OUTPUT_FILE = DATA_DIR / "cleaned_news.csv"

# Load the original dataset
df = pd.read_parquet(INPUT_FILE)

print("Original dataset shape:", df.shape)

# Keep only the columns needed for news detection
df = df[["title", "text", "label"]].copy()

# Replace missing title and text values
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["text"] = df["text"].fillna("").astype(str).str.strip()

# Remove records where both title and text are empty
df = df[(df["title"] != "") | (df["text"] != "")].copy()

# Ensure labels are numeric and valid
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df[df["label"].isin([0, 1])].copy()
df["label"] = df["label"].astype(int)

# Remove duplicate records
df = df.drop_duplicates(subset=["title", "text"])

# Sav

In [8]:
!pip install -q huggingface_hub pyarrow

from huggingface_hub import hf_hub_download
import os

data_path = hf_hub_download(
    repo_id="davanstrien/WELFake",
    filename="train-00000-of-00001-290868f0a36350c5.parquet",
    repo_type="dataset",
    local_dir="/content/FactCheck-AI/backend/data"
)

print("Dataset downloaded successfully!")
print("Location:", data_path)
print("File size:", round(os.path.getsize(data_path) / (1024**2), 2), "MB")

RemoteEntryNotFoundError: 404 Client Error. (Request ID: Root=1-6af15f4e-442f1aca2aefb27c6950f093;506256a9-e78f-434b-b24d-ae63ec3df852)

Entry Not Found for url: https://huggingface.co/datasets/davanstrien/WELFake/resolve/main/train-00000-of-00001-290868f0a36350c5.parquet.

In [9]:
from huggingface_hub import hf_hub_download
import os

data_path = hf_hub_download(
    repo_id="davanstrien/WELFake",
    filename="data/train-00000-of-00001-290868f0a36350c5.parquet",
    repo_type="dataset",
    local_dir="/content/FactCheck-AI/backend/data"
)

print("Dataset downloaded successfully!")
print("Location:", data_path)
print("File size:", round(os.path.getsize(data_path) / (1024**2), 2), "MB")

data/train-00000-of-00001-290868f0a36350(…): reconstructing file:   0%|          |  0.00B /  152MB            

data/train-00000-of-00001-290868f0a36350(…): downloading bytes:           |  0.00B            

Dataset downloaded successfully!
Location: /content/FactCheck-AI/backend/data/data/train-00000-of-00001-290868f0a36350c5.parquet
File size: 144.88 MB


In [10]:
import pandas as pd

dataset_path = "/content/FactCheck-AI/backend/data/data/train-00000-of-00001-290868f0a36350c5.parquet"

df = pd.read_parquet(dataset_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df[["title", "text", "label"]].isnull().sum())

print("\nLabel distribution:")
print(df["label"].value_counts().sort_index())

print("\nSample records:")
display(df.head(3))

Dataset shape: (72134, 3)

Columns:
['title', 'text', 'label']

Missing values:
title    558
text      39
label      0
dtype: int64

Label distribution:
label
0    35028
1    37106
Name: count, dtype: int64

Sample records:


,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,None,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1


In [11]:
import pandas as pd

# Keep required columns
df = df[["title", "text", "label"]].copy()

# Handle missing values
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["text"] = df["text"].fillna("").astype(str).str.strip()

# Remove completely empty records
df = df[(df["title"] != "") | (df["text"] != "")].copy()

# Validate labels
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df[df["label"].isin([0, 1])].copy()
df["label"] = df["label"].astype(int)

# Remove duplicates
df = df.drop_duplicates(subset=["title", "text"])

# Combine title and article text
df["content"] = (
    df["title"] + " " + df["text"]
).str.strip()

# Remove empty content
df = df[df["content"] != ""].copy()

df = df.reset_index(drop=True)

print("Cleaned dataset shape:", df.shape)
print("\nLabel distribution:")
print(df["label"].value_counts().sort_index())

print("\nSample:")
display(df[["content", "label"]].head(3))

Cleaned dataset shape: (63676, 4)

Label distribution:
label
0    34790
1    28886
Name: count, dtype: int64

Sample:


,content,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,1
1,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,1


In [12]:
from sklearn.model_selection import train_test_split

# First split: 80% training, 20% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

# Second split: divide temporary data equally
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Testing records:", len(test_df))

print("\nTraining distribution:")
print(train_df["label"].value_counts(normalize=True))

print("\nValidation distribution:")
print(val_df["label"].value_counts(normalize=True))

print("\nTesting distribution:")
print(test_df["label"].value_counts(normalize=True))

Training records: 50940
Validation records: 6368
Testing records: 6368

Training distribution:
label
0    0.546368
1    0.453632
Name: proportion, dtype: float64

Validation distribution:
label
0    0.546325
1    0.453675
Name: proportion, dtype: float64

Testing distribution:
label
0    0.546325
1    0.453675
Name: proportion, dtype: float64


In [13]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")
print("Model:", MODEL_NAME)
print("Maximum supported sequence length:", tokenizer.model_max_length)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully!
Model: distilbert-base-uncased
Maximum supported sequence length: 512


In [14]:
sample_text = train_df.iloc[0]["content"]

encoded = tokenizer(
    sample_text,
    truncation=True,
    padding="max_length",
    max_length=256,
    return_tensors="pt"
)

print("Original text:")
print(sample_text[:300])

print("\nTokenized input shape:")
print(encoded["input_ids"].shape)

print("\nAttention mask shape:")
print(encoded["attention_mask"].shape)

print("\nFirst 20 token IDs:")
print(encoded["input_ids"][0][:20])

Original text:
Internal BLM Memo Leaked: Trump’s Plan to Make America Great by Expediting Oil, Gas Production, Increasing Border Security, and Creating Jobs A Bureau of Land Management (BLM) internal working document that was leaked and reported on by the liberal Greenwire website spells out in detail President Do

Tokenized input shape:
torch.Size([1, 256])

Attention mask shape:
torch.Size([1, 256])

First 20 token IDs:
tensor([  101,  4722,  1038, 13728, 24443, 15748,  1024,  8398,  1521,  1055,
         2933,  2000,  2191,  2637,  2307,  2011,  4654,  5669, 15402,  3514])


In [15]:
from datasets import Dataset

MAX_LENGTH = 256

def tokenize_data(dataframe):
    dataset = Dataset.from_pandas(
        dataframe[["content", "label"]].reset_index(drop=True)
    )

    def tokenize_batch(batch):
        return tokenizer(
            batch["content"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH
        )

    dataset = dataset.map(
        tokenize_batch,
        batched=True,
        remove_columns=["content"]
    )

    dataset = dataset.rename_column("label", "labels")
    return dataset

train_dataset = tokenize_data(train_df)
val_dataset = tokenize_data(val_df)
test_dataset = tokenize_data(test_df)

print("Tokenization completed!")

print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Testing:", len(test_dataset))

print("\nAvailable columns:")
print(train_dataset.column_names)

Map:   0%|          | 0/50940 [00:00<?, ? examples/s]

Map:   0%|          | 0/6368 [00:00<?, ? examples/s]

Map:   0%|          | 0/6368 [00:00<?, ? examples/s]

Tokenization completed!
Training: 50940
Validation: 6368
Testing: 6368

Available columns:
['labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [16]:
from transformers import AutoModelForSequenceClassification
import torch

# Remove unused token_type_ids
for dataset_name in ["train_dataset", "val_dataset", "test_dataset"]:
    dataset = globals()[dataset_name]

    if "token_type_ids" in dataset.column_names:
        globals()[dataset_name] = dataset.remove_columns(
            ["token_type_ids"]
        )

# Load pretrained DistilBERT
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label={0: "FAKE", 1: "REAL"},
    label2id={"FAKE": 0, "REAL": 1}
)

print("Model loaded successfully!")
print("Model:", model.config.model_type)
print("Number of labels:", model.config.num_labels)
print("GPU available:", torch.cuda.is_available())
print("Training columns:", train_dataset.column_names)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!
Model: distilbert
Number of labels: 2
GPU available: True
Training columns: ['labels', 'input_ids', 'attention_mask']


In [17]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/factcheck-distilbert",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    logging_steps=100,
    save_total_limit=2,
    report_to="none",
    seed=42
)

print("Training configuration created successfully!")
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Learning rate:", training_args.learning_rate)
print("FP16:", training_args.fp16)

Training configuration created successfully!
Epochs: 3
Batch size: 8
Learning rate: 2e-05
FP16: True


In [18]:
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_metric.compute(
            predictions=predictions,
            references=labels
        )["accuracy"],

        "precision": precision_metric.compute(
            predictions=predictions,
            references=labels,
            average="weighted"
        )["precision"],

        "recall": recall_metric.compute(
            predictions=predictions,
            references=labels,
            average="weighted"
        )["recall"],

        "f1": f1_metric.compute(
            predictions=predictions,
            references=labels,
            average="weighted"
        )["f1"]
    }

print("Evaluation metrics configured successfully!")

Evaluation metrics configured successfully!


In [19]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("DistilBERT Trainer initialized successfully!")

DistilBERT Trainer initialized successfully!
